In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from src.core.pybullet_core import PybulletCore
from src.utils import *

# Open Pybullet GUI

In [ ]:
pb = PybulletCore()
pb.connect(robot_name = "indy7_v2", joint_limit=True, constraint_visualization = False)

# Move robot

In [ ]:
pb.MoveRobot([0, 30, -120, 0, -90, 0], degree=True, verbose=True)
from time import sleep
sleep(3)

# Inverse Kinematics (using XYZ position and ZYZ Euler angle)

In [ ]:
T_goal = xyzeul2SE3([0, -0.5, 0.5], [0,45,0], seq='ZYZ', degree=True)

pb.add_debug_frames([T_goal])
print(T_goal)

In [ ]:
q_i = pb.my_robot.q
qlist = np.zeros([6, 0])
qlist = np.concatenate((qlist, q_i), axis=1)
for _ in range(100):
    T_i = pb.my_robot.pinModel.FK(q_i)
    Jb_i = pb.my_robot.pinModel.Jb(q_i)

    R_i = T_i[0:3, 0:3]
    A_upper = np.concatenate((np.zeros([3, 3]), R_i), axis=1)
    A_lower = np.concatenate((np.eye(3), np.zeros([3, 3])), axis=1)
    A = np.concatenate((A_upper, A_lower), axis=0)

    Jv_i = A @ Jb_i
    
    r_err = T_goal[0:3, [3]] - T_i[0:3, [3]]

    # ==== IK using euler angle ====
    R_goal = T_goal[0:3, 0:3]
    euler_goal = Rot2eul(R_goal, seq='ZYZ', degree=True) * np.pi / 180
    euler_i = Rot2eul(R_i, seq='ZYZ', degree=True) * np.pi / 180

    phi_i, theta_i, psi_i = euler_i
    Tr_i = np.array([[0, -np.sin(phi_i), np.cos(phi_i)*np.sin(theta_i)],
                     [0, np.cos(phi_i), np.sin(phi_i)*np.sin(theta_i)],
                     [1, 0, np.cos(theta_i)]])
    Tr_i = np.linalg.pinv(Tr_i)
    

    euler_err = euler_goal - euler_i
    euler_err = np.array([euler_err]).T
    
    B_upper = np.concatenate((np.eye(3), np.zeros([3, 3])), axis=1)
    B_lower = np.concatenate((np.zeros([3, 3]), Tr_i), axis=1)
    B = np.concatenate((B_upper, B_lower), axis=0)

    Jr_i = B @ Jv_i

    p_err = np.concatenate((r_err, euler_err), axis=0)
 
    q_i = q_i + 0.1*np.linalg.pinv(Jr_i) @ p_err


    qlist = np.concatenate((qlist, q_i), axis=1)
    pb.MoveRobot(q_i, degree=False)
    sleep(0.1)


fig = plt.figure()
plt.plot(qlist.T*180/np.pi)
plt.legend(["q1", "q2", "q3", "q4", "q5", "q6"])
plt.xlabel("Step")
plt.ylabel("Joint angle (deg)")
plt.yticks([-180, -120, -60, 0, 60, 120, 180])
plt.show()

In [ ]:
T_goal = xyzeul2SE3([0.4, 0, 0.4], [0, 90, 0], seq='ZYZ', degree=True)

pb.add_debug_frames([T_goal])
print(T_goal)

In [ ]:
q_i = pb.my_robot.q
qlist = np.zeros([6, 0])
qlist = np.concatenate((qlist, q_i), axis=1)
for _ in range(100):
    T_i = pb.my_robot.pinModel.FK(q_i)
    Jb_i = pb.my_robot.pinModel.Jb(q_i)

    R_i = T_i[0:3, 0:3]
    A_upper = np.concatenate((np.zeros([3, 3]), R_i), axis=1)
    A_lower = np.concatenate((np.eye(3), np.zeros([3, 3])), axis=1)
    A = np.concatenate((A_upper, A_lower), axis=0)

    Jv_i = A @ Jb_i
    
    r_err = T_goal[0:3, [3]] - T_i[0:3, [3]]

    # ==== IK using euler angle ====
    R_goal = T_goal[0:3, 0:3]
    euler_goal = Rot2eul(R_goal, seq='ZYZ', degree=True) * np.pi / 180
    euler_i = Rot2eul(R_i, seq='ZYZ', degree=True) * np.pi / 180

    phi_i, theta_i, psi_i = euler_i
    Tr_i = np.array([[0, -np.sin(phi_i), np.cos(phi_i)*np.sin(theta_i)],
                     [0, np.cos(phi_i), np.sin(phi_i)*np.sin(theta_i)],
                     [1, 0, np.cos(theta_i)]])
    Tr_i = np.linalg.pinv(Tr_i)
    

    euler_err = euler_goal - euler_i
    euler_err = np.array([euler_err]).T
    
    B_upper = np.concatenate((np.eye(3), np.zeros([3, 3])), axis=1)
    B_lower = np.concatenate((np.zeros([3, 3]), Tr_i), axis=1)
    B = np.concatenate((B_upper, B_lower), axis=0)

    Jr_i = B @ Jv_i

    p_err = np.concatenate((r_err, euler_err), axis=0)
 
    q_i = q_i + 0.1*np.linalg.pinv(Jr_i) @ p_err


    qlist = np.concatenate((qlist, q_i), axis=1)
    pb.MoveRobot(q_i, degree=False)
    sleep(0.1)


fig = plt.figure()
plt.plot(qlist.T*180/np.pi)
plt.legend(["q1", "q2", "q3", "q4", "q5", "q6"])
plt.xlabel("Step")
plt.ylabel("Joint angle (deg)")
plt.yticks([-180, -120, -60, 0, 60, 120, 180])
plt.show()

# Moving through a circular trajectory


Circle centered at $(x,y,z)=(0.4, 0, 0.4)$ with radius $R=0.1$

In [ ]:
num_points = 100
r_goal_array = [[0.4, 0 + 0.1*np.cos(2*np.pi/num_points*i), 0.4 + 0.1*np.sin(2*np.pi/num_points*i)] for i in range(num_points)]

for i in range(1000):
    T_goal = xyzeul2SE3(r_goal_array[i%num_points], [0, 90, 0], seq='ZYZ', degree=True)
    for _ in range(10):
        T_i = pb.my_robot.pinModel.FK(q_i)
        Jb_i = pb.my_robot.pinModel.Jb(q_i)

        R_i = T_i[0:3, 0:3]
        A_upper = np.concatenate((np.zeros([3, 3]), R_i), axis=1)
        A_lower = np.concatenate((np.eye(3), np.zeros([3, 3])), axis=1)
        A = np.concatenate((A_upper, A_lower), axis=0)

        Jv_i = A @ Jb_i
        
        r_err = T_goal[0:3, [3]] - T_i[0:3, [3]]


        # ==== IK using exponential coordinate ====
        R_err = T_i[0:3, 0:3].T @ T_goal[0:3, 0:3]
        xi_err = Rot2Vec(R_err)
        p_err = np.concatenate((r_err, xi_err), axis=0)
    
        q_i = q_i + 0.1*np.linalg.pinv(Jv_i) @ p_err


        qlist = np.concatenate((qlist, q_i), axis=1)
        pb.MoveRobot(q_i, degree=False)
        sleep(0.01)


Exception in thread Thread-8 (_thread_main):
Traceback (most recent call last):
  File "c:\Users\hsjung02\AppData\Local\miniconda3\envs\MECH439\lib\threading.py", line 1016, in _bootstrap_inner
    self.run()
  File "c:\Users\hsjung02\AppData\Local\miniconda3\envs\MECH439\lib\threading.py", line 953, in run
    self._target(*self._args, **self._kwargs)
  File "c:\Users\hsjung02\OneDrive - postech.ac.kr\Desktop\2026-1 로보틱스개론\실습\code\src\core\pybullet_core.py", line 123, in _thread_main
    self.my_robot.robot_update()
  File "c:\Users\hsjung02\OneDrive - postech.ac.kr\Desktop\2026-1 로보틱스개론\실습\code\src\core\pybullet_robot.py", line 57, in robot_update
    self._get_robot_states()      # update robot's states
  File "c:\Users\hsjung02\OneDrive - postech.ac.kr\Desktop\2026-1 로보틱스개론\실습\code\src\core\pybullet_robot.py", line 297, in _get_robot_states
    states = p.getJointState(self.robotId, idx, physicsClientId=self.ClientId)
pybullet.error: Not connected to physics server.


In [ ]:
pb.disconnect()